In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import math

# ------------------------------
# Helper functions
# ------------------------------

def load_predictions(csv_path):
    """Load predictions_test.csv and compute conf, pred_labels, correct."""
    df = pd.read_csv(csv_path)
    class_cols = ["p_glioma", "p_meningioma", "p_pituitary"]
    classes = ["glioma", "meningioma", "pituitary"]

    probs = df[class_cols].values
    confidences = probs.max(axis=1)
    pred_idx = probs.argmax(axis=1)
    pred_labels = [classes[i] for i in pred_idx]
    true_idx = df["true"].map({c: i for i, c in enumerate(classes)}).values
    true_probs = probs[np.arange(len(df)), true_idx]

    df["conf"] = confidences
    df["pred_from_probs"] = pred_labels
    df["correct"] = (df["pred_from_probs"] == df["true"]).astype(int)
    df["true_prob"] = true_probs

    return df


def compute_ece(df, n_bins=10):
    """Compute Expected Calibration Error."""
    confidences = df["conf"].values
    correct = df["correct"].values

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidences, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    total_n = len(df)
    ece = 0.0

    for b in range(n_bins):
        sub = df[bin_ids == b]
        if len(sub) == 0:
            continue
        acc = sub["correct"].mean()
        avg_conf = sub["conf"].mean()
        ece += (len(sub) / total_n) * abs(acc - avg_conf)

    return ece


def compute_nll(df):
    """Compute mean negative log-likelihood."""
    eps = 1e-12
    nll = -np.log(np.clip(df["true_prob"].values, eps, 1.0))
    return nll.mean()


def evaluate_predictions(csv_path):
    """Compute Accuracy, macro-F1, ECE, NLL for one fold."""
    df = load_predictions(csv_path)

    acc = accuracy_score(df["true"], df["pred_from_probs"])
    f1 = f1_score(df["true"], df["pred_from_probs"], average='macro')
    ece = compute_ece(df)
    nll = compute_nll(df)

    return acc, f1, ece, nll


def evaluate_all_folds(base_dir):
    """Evaluate fold_01 ... fold_05 inside base_dir."""
    results = []
    for k in range(1, 6):
        path = os.path.join(base_dir, f"fold_{k:02d}", "predictions_test.csv")
        if os.path.exists(path):
            acc, f1, ece, nll = evaluate_predictions(path)
            results.append({
                "fold": k,
                "accuracy": acc,
                "macro_f1": f1,
                "ece": ece,
                "nll": nll
            })
        else:
            print(f"WARNING: File missing: {path}")
    return pd.DataFrame(results)


def evaluate_ensemble(csv_path):
    """Evaluate ensemble predictions."""
    acc, f1, ece, nll = evaluate_predictions(csv_path)
    return pd.DataFrame([{
        "fold": "ensemble",
        "accuracy": acc,
        "macro_f1": f1,
        "ece": ece,
        "nll": nll
    }])


In [3]:
base_dir_inge = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold"

df_folds_inge = evaluate_all_folds(base_dir_inge)
print(df_folds_inge)

df_ensemble_inge = evaluate_ensemble(
    "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold/test_ensemble_predictions.csv"
)
print(df_ensemble_inge)


   fold  accuracy  macro_f1       ece       nll
0     1  0.960114  0.953403  0.020740  0.128097
1     2  0.944444  0.935401  0.024481  0.185428
2     3  0.960114  0.954299  0.017477  0.107009
3     4  0.945869  0.935612  0.030455  0.155367
4     5  0.957265  0.950356  0.017741  0.100176
       fold  accuracy  macro_f1       ece       nll
0  ensemble  0.960114  0.953784  0.004712  0.090848


In [4]:
base_dir_seed122 = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold"

df_folds_seed122 = evaluate_all_folds(base_dir_seed122)
print(df_folds_seed122)

df_ensemble_seed122 = evaluate_ensemble(
    "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold/test_ensemble_predictions.csv"
)
print(df_ensemble_seed122)


   fold  accuracy  macro_f1       ece       nll
0     1  0.948408  0.942060  0.031528  0.168424
1     2  0.944018  0.937321  0.036562  0.209396
2     3  0.948408  0.943760  0.033309  0.196576
3     4  0.947311  0.938742  0.028313  0.199095
4     5  0.950604  0.944681  0.033357  0.197044
       fold  accuracy  macro_f1       ece       nll
0  ensemble  0.956092  0.951147  0.014745  0.127756


In [9]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

CLASSES = ["glioma", "meningioma", "pituitary"]
CLASS_COLS = ["p_glioma", "p_meningioma", "p_pituitary"]


def load_predictions(csv_path):
    """Load predictions_test.csv and add predicted label column."""
    df = pd.read_csv(csv_path)

    probs = df[CLASS_COLS].values
    pred_idx = probs.argmax(axis=1)
    pred_labels = [CLASSES[i] for i in pred_idx]

    df["pred"] = pred_labels
    return df


def compute_ece_from_arrays(confidences, correct, n_bins=10):
    """Compute ECE for given confidences and correctness (0/1)."""
    confidences = np.asarray(confidences)
    correct = np.asarray(correct)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidences, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    total_n = len(confidences)
    ece = 0.0

    for b in range(n_bins):
        mask = (bin_ids == b)
        if not np.any(mask):
            continue
        acc_bin = correct[mask].mean()
        conf_bin = confidences[mask].mean()
        ece += (mask.sum() / total_n) * abs(acc_bin - conf_bin)

    return ece


def classwise_metrics(df, class_name, prob_col, n_bins=10):
    """
    Calcula F1, ACC, ECE y NLL para una clase específica.
    - F1: one-vs-rest (clase vs no clase)
    - ACC (por clase): aciertos / muestras con true == clase (recall)
    - ECE: usando p_class como confianza y correct = 1 si true == clase, 0 si no
    - NLL: -log(p_class) solo sobre muestras con true == clase
    """
    # One-vs-rest para F1
    y_true_bin = (df["true"] == class_name).astype(int)
    y_pred_bin = (df["pred"] == class_name).astype(int)
    f1 = f1_score(y_true_bin, y_pred_bin)

    # Accuracy por clase (recall): aciertos dentro de esa clase
    mask_true = (df["true"] == class_name)
    if mask_true.sum() > 0:
        acc = (df.loc[mask_true, "pred"] == class_name).mean()
    else:
        acc = np.nan

    # ECE por clase: p_class como confianza, correct = 1 si true == clase
    probs_class = df[prob_col].values
    correct_for_ece = (df["true"] == class_name).astype(int).values
    ece = compute_ece_from_arrays(probs_class, correct_for_ece, n_bins=n_bins)

    # NLL por clase: solo donde true == clase
    eps = 1e-12
    if mask_true.sum() > 0:
        p_true_class = np.clip(df.loc[mask_true, prob_col].values, eps, 1.0)
        nll = -np.log(p_true_class).mean()
    else:
        nll = np.nan

    return f1, acc, ece, nll


def evaluate_model(csv_path, n_bins=10):
    """
    Devuelve un dict con F1, ACC, ECE, NLL por clase para un solo modelo (fold o ensemble).
    """
    df = load_predictions(csv_path)

    metrics = {}
    for cls, col in zip(CLASSES, CLASS_COLS):
        f1, acc, ece, nll = classwise_metrics(df, cls, col, n_bins=n_bins)
        metrics[f"f1_{cls}"] = f1
        metrics[f"acc_{cls}"] = acc
        metrics[f"ece_{cls}"] = ece
        metrics[f"nll_{cls}"] = nll

    return metrics


def evaluate_folds_and_ensemble(base_dir, ensemble_path, n_bins=10):
    """
    Calcula métricas por clase para:
    - 5 folds (fold_01 ... fold_05)
    - ensemble
    Devuelve dos DataFrames:
    - df_summary_classes: filas = clases, columnas = métricas de folds (mean) + ensemble
    - df_folds_raw: opcional, filas = folds, columnas = métricas por clase (para debug)
    """
    # 1) folds
    fold_metrics = []
    for k in range(1, 6):
        path = os.path.join(base_dir, f"fold_{k:02d}", "predictions_test.csv")
        if not os.path.exists(path):
            print(f"WARNING: missing {path}")
            continue
        m = evaluate_model(path, n_bins=n_bins)
        m["fold"] = k
        fold_metrics.append(m)

    df_folds = pd.DataFrame(fold_metrics)

    # 2) promedio de folds por clase
    mean_metrics = df_folds.mean(numeric_only=True)

    # 3) ensemble
    ensemble_metrics = evaluate_model(ensemble_path, n_bins=n_bins)

    # 4) construir tabla por clase
    rows = []
    for cls in CLASSES:
        row = {
            "class": cls,
            "f1_folds_mean": mean_metrics[f"f1_{cls}"],
            "acc_folds_mean": mean_metrics[f"acc_{cls}"],
            "ece_folds_mean": mean_metrics[f"ece_{cls}"],
            "nll_folds_mean": mean_metrics[f"nll_{cls}"],
            "f1_ensemble": ensemble_metrics[f"f1_{cls}"],
            "acc_ensemble": ensemble_metrics[f"acc_{cls}"],
            "ece_ensemble": ensemble_metrics[f"ece_{cls}"],
            "nll_ensemble": ensemble_metrics[f"nll_{cls}"],
        }
        rows.append(row)

    df_summary_classes = pd.DataFrame(rows)
    return df_summary_classes, df_folds


In [10]:
# === Reduced set ===
base_dir_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold"
ensemble_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold/test_ensemble_predictions.csv"

df_reduced_classes, df_reduced_folds = evaluate_folds_and_ensemble(
    base_dir_reduced,
    ensemble_reduced,
    n_bins=10
)

print("=== REDUCED SET: per-class metrics (folds mean + ensemble) ===")
print(df_reduced_classes)


# === Complete set ===
base_dir_complete = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold"
ensemble_complete = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold/test_ensemble_predictions.csv"

df_complete_classes, df_complete_folds = evaluate_folds_and_ensemble(
    base_dir_complete,
    ensemble_complete,
    n_bins=10
)

print("\n=== COMPLETE SET: per-class metrics (folds mean + ensemble) ===")
print(df_complete_classes)


=== REDUCED SET: per-class metrics (folds mean + ensemble) ===
        class  f1_folds_mean  acc_folds_mean  ece_folds_mean  nll_folds_mean  \
0      glioma       0.957565        0.953846        0.021488        0.119479   
1  meningioma       0.900486        0.919149        0.026001        0.295255   
2   pituitary       0.979392        0.972691        0.010454        0.064308   

   f1_ensemble  acc_ensemble  ece_ensemble  nll_ensemble  
0     0.960000      0.961538      0.015344      0.086175  
1     0.915493      0.921986      0.017573      0.174515  
2     0.985859      0.979920      0.012506      0.049326  

=== COMPLETE SET: per-class metrics (folds mean + ensemble) ===
        class  f1_folds_mean  acc_folds_mean  ece_folds_mean  nll_folds_mean  \
0      glioma       0.953350        0.953363        0.031677        0.166559   
1  meningioma       0.892094        0.895876        0.031277        0.415854   
2   pituitary       0.978495        0.975646        0.009741        0.08070

In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

CLASSES = ["glioma", "meningioma", "pituitary"]
CLASS_COLS = ["p_glioma", "p_meningioma", "p_pituitary"]


# ------------------------
# Basic helpers
# ------------------------

def load_predictions(path):
    """Load predictions_test.csv and add pred + conf columns."""
    df = pd.read_csv(path)
    probs = df[CLASS_COLS].values
    pred_idx = probs.argmax(axis=1)
    df["pred"] = [CLASSES[i] for i in pred_idx]
    df["conf"] = probs.max(axis=1)
    return df


def compute_ece(conf, correct, n_bins=10):
    """Expected Calibration Error using given confidences and correctness (0/1)."""
    conf = np.asarray(conf)
    correct = np.asarray(correct)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    N = len(conf)
    ece = 0.0
    for b in range(n_bins):
        mask = (bin_ids == b)
        if not np.any(mask):
            continue
        acc_bin = correct[mask].mean()
        conf_bin = conf[mask].mean()
        ece += (mask.sum() / N) * abs(acc_bin - conf_bin)
    return ece


def compute_metrics_for_df(df):
    """
    Compute metrics for:
      - 'global'
      - each class in CLASSES

    Returns:
      dict: key -> 'global' or class name,
            value -> dict with acc, f1, ece, nll, conf
    """
    results = {}

    # ---- Global metrics ----
    # ACC global
    acc_g = accuracy_score(df["true"], df["pred"])
    # F1 macro
    f1_g = f1_score(df["true"], df["pred"], average="macro")
    # ECE global: conf = max prob, correct = (pred == true)
    correct_g = (df["pred"] == df["true"]).astype(int).values
    ece_g = compute_ece(df["conf"].values, correct_g)
    # NLL global: prob of true class
    true_probs = []
    for _, row in df.iterrows():
        p_true = row["p_" + row["true"]]
        true_probs.append(p_true)
    true_probs = np.clip(np.array(true_probs), 1e-12, 1.0)
    nll_g = -np.log(true_probs).mean()
    # Conf global: promedio de conf (max prob)
    conf_g = df["conf"].mean()

    results["global"] = {
        "acc": acc_g,
        "f1": f1_g,
        "ece": ece_g,
        "nll": nll_g,
        "conf": conf_g,
    }

    # ---- Per-class metrics ----
    for cls, col in zip(CLASSES, CLASS_COLS):
        mask_true = (df["true"] == cls)

        # ACC por clase = recall de esa clase
        if mask_true.sum() > 0:
            acc_c = (df.loc[mask_true, "pred"] == cls).mean()
            # NLL y prob media solo en samples de esa clase
            p_cls_true = np.clip(df.loc[mask_true, col].values, 1e-12, 1.0)
            nll_c = -np.log(p_cls_true).mean()
            conf_c = p_cls_true.mean()
        else:
            acc_c = np.nan
            nll_c = np.nan
            conf_c = np.nan

        # F1 one-vs-rest
        y_true_bin = mask_true.astype(int)
        y_pred_bin = (df["pred"] == cls).astype(int)
        f1_c = f1_score(y_true_bin, y_pred_bin)

        # ECE por clase:
        # conf = p_cls (para todos), correct = 1 si true == cls, 0 si no
        conf_all = df[col].values
        correct_all = mask_true.astype(int).values
        ece_c = compute_ece(conf_all, correct_all)

        results[cls] = {
            "acc": acc_c,
            "f1": f1_c,
            "ece": ece_c,
            "nll": nll_c,
            "conf": conf_c,
        }

    return results


# ------------------------
# Folds + ensemble block
# ------------------------

def evaluate_block(base_dir, ensemble_path, n_folds=5):
    """
    Evalúa 5 folds + ensemble y devuelve:
      - fold_summary: media y std sobre folds para cada (global + clase)
      - ensemble_summary: métricas del ensemble para cada (global + clase)
    """
    fold_metrics_list = []

    # --- Folds ---
    for k in range(1, n_folds + 1):
        path = os.path.join(base_dir, f"fold_{k:02d}", "predictions_test.csv")
        if not os.path.exists(path):
            print(f"WARNING: missing {path}")
            continue
        df = load_predictions(path)
        metrics = compute_metrics_for_df(df)
        fold_metrics_list.append(metrics)

    # Estructura: fold_metrics_list[f][key]['metric']

    # Build fold summary (mean, std) por key (global + clases) y metric
    rows_fold = []
    for key in ["global"] + CLASSES:
        # para cada métrica
        row = {"class": key}
        for metric in ["acc", "f1", "ece", "nll", "conf"]:
            vals = [fm[key][metric] for fm in fold_metrics_list]
            vals = np.array(vals, dtype=float)
            mean = vals.mean()
            std = vals.std(ddof=1)  # sample std
            row[f"{metric}_mean"] = mean
            row[f"{metric}_std"] = std
        rows_fold.append(row)

    fold_summary = pd.DataFrame(rows_fold)

    # --- Ensemble ---
    df_ens = load_predictions(ensemble_path)
    ens_metrics = compute_metrics_for_df(df_ens)

    rows_ens = []
    for key in ["global"] + CLASSES:
        row = {"class": key}
        for metric in ["acc", "f1", "ece", "nll", "conf"]:
            row[metric] = ens_metrics[key][metric]
        rows_ens.append(row)

    ensemble_summary = pd.DataFrame(rows_ens)

    # Redondear a 3 decimales para comodidad
    fold_summary = fold_summary.round(3)
    ensemble_summary = ensemble_summary.round(3)

    return fold_summary, ensemble_summary


In [10]:

# REDUCED
base_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge0.8fold"
ens_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge0.8fold/test_ensemble_predictions.csv"

fold_reduced, ens_reduced_df = evaluate_block(base_reduced, ens_reduced)

print("\n=== REDUCED – fold mean/std ===")
print(fold_reduced)
print("\n=== REDUCED – ensemble ===")
print(ens_reduced_df)



=== REDUCED – fold mean/std ===
        class  acc_mean  acc_std  f1_mean  f1_std  ece_mean  ece_std  \
0      global     0.950    0.007    0.942   0.007     0.028    0.005   
1      glioma     0.947    0.006    0.955   0.005     0.027    0.003   
2  meningioma     0.916    0.012    0.889   0.009     0.031    0.006   
3   pituitary     0.974    0.013    0.981   0.010     0.011    0.003   

   nll_mean  nll_std  conf_mean  conf_std  
0     0.146    0.027      0.975     0.002  
1     0.158    0.017      0.941     0.004  
2     0.265    0.071      0.907     0.006  
3     0.064    0.030      0.964     0.014  

=== REDUCED – ensemble ===
        class    acc     f1    ece    nll   conf
0      global  0.954  0.946  0.009  0.100  0.963
1      glioma  0.946  0.958  0.016  0.101  0.941
2  meningioma  0.922  0.893  0.018  0.187  0.907
3   pituitary  0.984  0.986  0.013  0.050  0.964


In [5]:

# REDUCED
base_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.0fold"
ens_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.0fold/test_ensemble_predictions.csv"

fold_reduced, ens_reduced_df = evaluate_block(base_reduced, ens_reduced)

print("\n=== REDUCED – fold mean/std ===")
print(fold_reduced)
print("\n=== REDUCED – ensemble ===")
print(ens_reduced_df)



=== REDUCED – fold mean/std ===
        class  acc_mean  acc_std  f1_mean  f1_std  ece_mean  ece_std  \
0      global     0.952    0.011    0.944   0.014     0.025    0.009   
1      glioma     0.951    0.018    0.957   0.005     0.022    0.005   
2  meningioma     0.916    0.023    0.896   0.031     0.027    0.011   
3   pituitary     0.973    0.012    0.978   0.008     0.013    0.002   

   nll_mean  nll_std  conf_mean  conf_std  
0     0.142    0.032      0.972     0.002  
1     0.144    0.057      0.943     0.017  
2     0.264    0.109      0.904     0.022  
3     0.070    0.037      0.964     0.013  

=== REDUCED – ensemble ===
        class    acc     f1    ece    nll   conf
0      global  0.953  0.944  0.015  0.093  0.959
1      glioma  0.952  0.960  0.015  0.092  0.943
2  meningioma  0.915  0.896  0.016  0.173  0.904
3   pituitary  0.976  0.978  0.011  0.049  0.964


In [11]:

# REDUCED
base_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.1fold"
ens_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.1fold/test_ensemble_predictions.csv"

fold_reduced, ens_reduced_df = evaluate_block(base_reduced, ens_reduced)

print("\n=== REDUCED – fold mean/std ===")
print(fold_reduced)
print("\n=== REDUCED – ensemble ===")
print(ens_reduced_df)



=== REDUCED – fold mean/std ===
        class  acc_mean  acc_std  f1_mean  f1_std  ece_mean  ece_std  \
0      global     0.953    0.010    0.945   0.012     0.024    0.009   
1      glioma     0.946    0.014    0.957   0.009     0.026    0.007   
2  meningioma     0.926    0.026    0.898   0.024     0.030    0.009   
3   pituitary     0.978    0.002    0.981   0.004     0.009    0.003   

   nll_mean  nll_std  conf_mean  conf_std  
0     0.137    0.038      0.974     0.004  
1     0.149    0.051      0.940     0.013  
2     0.251    0.110      0.917     0.021  
3     0.058    0.010      0.965     0.003  

=== REDUCED – ensemble ===
        class    acc     f1    ece    nll   conf
0      global  0.956  0.948  0.012  0.096  0.964
1      glioma  0.942  0.958  0.017  0.104  0.940
2  meningioma  0.936  0.901  0.017  0.167  0.917
3   pituitary  0.984  0.986  0.010  0.046  0.965


In [6]:

# REDUCED
base_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.2fold"
ens_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.2fold/test_ensemble_predictions.csv"

fold_reduced, ens_reduced_df = evaluate_block(base_reduced, ens_reduced)

print("\n=== REDUCED – fold mean/std ===")
print(fold_reduced)
print("\n=== REDUCED – ensemble ===")
print(ens_reduced_df)



=== REDUCED – fold mean/std ===
        class  acc_mean  acc_std  f1_mean  f1_std  ece_mean  ece_std  \
0      global     0.953    0.009    0.944   0.011     0.025    0.006   
1      glioma     0.949    0.016    0.959   0.008     0.023    0.007   
2  meningioma     0.922    0.005    0.893   0.021     0.030    0.009   
3   pituitary     0.975    0.010    0.981   0.006     0.010    0.003   

   nll_mean  nll_std  conf_mean  conf_std  
0     0.138    0.039      0.976     0.004  
1     0.159    0.049      0.944     0.013  
2     0.220    0.057      0.915     0.008  
3     0.064    0.034      0.965     0.016  

=== REDUCED – ensemble ===
        class    acc     f1    ece    nll   conf
0      global  0.960  0.953  0.009  0.094  0.963
1      glioma  0.952  0.963  0.013  0.107  0.944
2  meningioma  0.936  0.910  0.016  0.149  0.915
3   pituitary  0.984  0.986  0.010  0.047  0.965


In [7]:

# REDUCED
base_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold"
ens_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold/test_ensemble_predictions.csv"

fold_reduced, ens_reduced_df = evaluate_block(base_reduced, ens_reduced)

print("\n=== REDUCED – fold mean/std ===")
print(fold_reduced)
print("\n=== REDUCED – ensemble ===")
print(ens_reduced_df)



=== REDUCED – fold mean/std ===
        class  acc_mean  acc_std  f1_mean  f1_std  ece_mean  ece_std  \
0      global     0.954    0.008    0.946   0.010     0.022    0.005   
1      glioma     0.954    0.009    0.958   0.007     0.021    0.005   
2  meningioma     0.919    0.038    0.900   0.019     0.026    0.006   
3   pituitary     0.973    0.005    0.979   0.005     0.010    0.001   

   nll_mean  nll_std  conf_mean  conf_std  
0     0.135    0.035      0.973     0.006  
1     0.119    0.022      0.948     0.006  
2     0.295    0.189      0.906     0.043  
3     0.064    0.007      0.964     0.004  

=== REDUCED – ensemble ===
        class    acc     f1    ece    nll   conf
0      global  0.960  0.954  0.005  0.091  0.961
1      glioma  0.962  0.960  0.015  0.086  0.948
2  meningioma  0.922  0.915  0.018  0.175  0.906
3   pituitary  0.980  0.986  0.013  0.049  0.964


In [8]:

# REDUCED
base_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.4fold"
ens_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.4fold/test_ensemble_predictions.csv"

fold_reduced, ens_reduced_df = evaluate_block(base_reduced, ens_reduced)

print("\n=== REDUCED – fold mean/std ===")
print(fold_reduced)
print("\n=== REDUCED – ensemble ===")
print(ens_reduced_df)



=== REDUCED – fold mean/std ===
        class  acc_mean  acc_std  f1_mean  f1_std  ece_mean  ece_std  \
0      global     0.950    0.005    0.942   0.007     0.024    0.006   
1      glioma     0.954    0.007    0.956   0.004     0.025    0.006   
2  meningioma     0.906    0.015    0.894   0.017     0.024    0.005   
3   pituitary     0.971    0.013    0.976   0.005     0.013    0.002   

   nll_mean  nll_std  conf_mean  conf_std  
0     0.142    0.019      0.970     0.005  
1     0.123    0.018      0.946     0.006  
2     0.309    0.094      0.893     0.018  
3     0.072    0.034      0.960     0.016  

=== REDUCED – ensemble ===
        class    acc     f1    ece    nll   conf
0      global  0.953  0.945  0.015  0.101  0.959
1      glioma  0.952  0.958  0.013  0.087  0.946
2  meningioma  0.915  0.899  0.015  0.215  0.893
3   pituitary  0.976  0.978  0.014  0.053  0.960


In [14]:
# COMPLETE
base_complete = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold"
ens_complete = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold/test_ensemble_predictions.csv"

fold_complete, ens_complete_df = evaluate_block(base_complete, ens_complete)

print("=== COMPLETE – fold mean/std ===")
print(fold_complete)
print("\n=== COMPLETE – ensemble ===")
print(ens_complete_df)


# REDUCED
base_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold"
ens_reduced = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold/test_ensemble_predictions.csv"

fold_reduced, ens_reduced_df = evaluate_block(base_reduced, ens_reduced)

print("\n=== REDUCED – fold mean/std ===")
print(fold_reduced)
print("\n=== REDUCED – ensemble ===")
print(ens_reduced_df)


=== COMPLETE – fold mean/std ===
        class  acc_mean  acc_std  f1_mean  f1_std  ece_mean  ece_std  \
0      global     0.948    0.002    0.941   0.003     0.033    0.003   
1      glioma     0.953    0.015    0.953   0.003     0.032    0.003   
2  meningioma     0.896    0.030    0.892   0.011     0.031    0.005   
3   pituitary     0.976    0.014    0.978   0.007     0.010    0.004   

   nll_mean  nll_std  conf_mean  conf_std  
0     0.194    0.015      0.978     0.002  
1     0.167    0.080      0.949     0.015  
2     0.416    0.145      0.882     0.028  
3     0.081    0.052      0.970     0.013  

=== COMPLETE – ensemble ===
        class    acc     f1    ece    nll   conf
0      global  0.956  0.951  0.015  0.128  0.965
1      glioma  0.964  0.959  0.018  0.095  0.949
2  meningioma  0.907  0.910  0.020  0.307  0.882
3   pituitary  0.978  0.985  0.007  0.054  0.970

=== REDUCED – fold mean/std ===
        class  acc_mean  acc_std  f1_mean  f1_std  ece_mean  ece_std  \
0      